# Task 01 Solution: LCEL Classification Chain

## Setup

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

OPENAI_API_KEY = "your-api-key-here"
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY)
print("✓ LLM initialized")


In [ ]:
import json, os

# In Jupyter, os.getcwd() returns the notebook's directory (learning/, tasks/, solutions/)
# Fixtures are one level up at ../fixtures/input/
fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))

with open(os.path.join(fixtures, "tickets.json")) as f:
    tickets = json.load(f)

with open(os.path.join(fixtures, "knowledge_base.json")) as f:
    knowledge_base = json.load(f)

with open(os.path.join(fixtures, "test_questions.json")) as f:
    test_questions = json.load(f)

print(f"✓ Loaded {len(tickets)} tickets, {len(knowledge_base)} KB articles, {len(test_questions)} test questions")


## Part 1: Classification Chain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are a support ticket classifier. "
        "Return ONLY valid JSON with exactly two keys:\n"
        '  "category": one of billing | technical | account | shipping\n'
        '  "priority": one of high | medium | low\n\n'
        "Priority definitions:\n"
        "  high: financial damage (duplicate/unauthorized charge), production system down, "
        "security compromise (hacked account), data loss\n"
        "  medium: non-critical bugs with workarounds, missing documents (invoices), "
        "degraded performance, account access issues\n"
        "  low: informational/how-to questions, cosmetic bugs, feature preferences\n\n"
        "No markdown, no code blocks, no explanation."
    )),
    ("human", "{ticket}"),
])

chain = prompt | llm | JsonOutputParser()

test = chain.invoke({"ticket": "I was charged twice"})
print("Test result:", test)


In [ ]:
assert hasattr(chain, 'invoke')
print("✓ chain has .invoke()")


## Part 2: Batch Classification

In [ ]:
def classify_all(chain, tickets):
    inputs = [{"ticket": t["text"]} for t in tickets]
    return chain.batch(inputs)

results = classify_all(chain, tickets)


In [ ]:
VALID_CATEGORIES = {"billing", "technical", "account", "shipping"}
VALID_PRIORITIES = {"high", "medium", "low"}

for i, r in enumerate(results):
    assert isinstance(r, dict)
    assert "category" in r
    assert "priority" in r
    assert r["category"] in VALID_CATEGORIES
    assert r["priority"] in VALID_PRIORITIES

print(f"✓ All {len(results)} results valid")


In [ ]:
correct_cat = sum(1 for r, t in zip(results, tickets) if r.get("category") == t["category"])
correct_pri = sum(1 for r, t in zip(results, tickets) if r.get("priority") == t["priority"])

cat_acc = correct_cat / len(tickets)
pri_acc = correct_pri / len(tickets)
print(f"Category: {cat_acc:.0%}, Priority: {pri_acc:.0%}")

for r, t in zip(results, tickets):
    if r.get("category") != t["category"] or r.get("priority") != t["priority"]:
        print(f"  ID {t['id']}: got {r} | expected {t['category']}/{t['priority']}")

assert cat_acc >= 0.80, f"Category {cat_acc:.0%} < 80%"
assert pri_acc >= 0.60, f"Priority {pri_acc:.0%} < 60%"
print("✓ Accuracy targets met!")


## Part 3: CoT Chain

In [ ]:
cot_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are a support ticket classifier. Think step by step, then classify.\n"
        'Return ONLY valid JSON: {{"reasoning": "brief explanation", '
        '"category": "billing|technical|account|shipping", "priority": "high|medium|low"}}'
    )),
    ("human", "{ticket}"),
])

cot_chain = cot_prompt | llm | JsonOutputParser()

sample_result = cot_chain.invoke({"ticket": "I was charged twice this billing cycle."})
print(f"Reasoning: {sample_result['reasoning'][:80]}...")
print(f"Category: {sample_result['category']}, Priority: {sample_result['priority']}")


In [ ]:
assert "reasoning" in sample_result
assert len(sample_result["reasoning"]) > 20
print("✓ CoT chain format correct")
